
IMP Scripps Workshop Mar 2026
=============================

Here we demonstrate basic usage of the IMP software for integrative modeling.

First, we need to install the IMP software:

In [ ]:
!add-apt-repository -y ppa:salilab/ppa
!apt install imp
import sys, os, glob
sys.path.append(os.path.dirname(glob.glob('/usr/lib/python*/dist-packages/IMP')[0]))
os.environ['PYTHONPATH'] = os.path.dirname(glob.glob('/usr/lib/python*/dist-packages/IMP')[0])

# IMP C++/Python library

The main way to use the IMP C++/Python library is to write a Python script. Below we'll look at a simple example.

## Imports

To use IMP, first we must make IMP classes in the IMP kernel (`IMP`) and the `IMP.algebra` and `IMP.core` modules available. IMP is divided into modules that combine related functionality, such as handling of a particular type of experimental data.

In [ ]:
import IMP
import IMP.algebra
import IMP.core

## Model and particles

Next, we create a new Model object (an instance of the Model class) and assign it to the variable ‘m’. An IMP Model is a container that holds knowledge of the entire system (see the IMP Reference Guide).

Then we create two particles called ‘p1’ and ‘p2’; each is an abstract data container inside the model (really p1 and p2 are just indices into a data structure inside Model) and can hold any number of attribute:value pairs, e.g.
- xyz coordinates
- mass
- radius
- pointers to other particles, to represent a bond (two other particles), or hierarchy (parents, children)
- element, residue/atom name, etc.


In [ ]:
m = IMP.Model()
# Create two "untyped" particles
p1 = m.add_particle('p1')
p2 = m.add_particle('p2')

## Decorators

A *decorator* lets us use a specific set of functionality on a particle. ‘d1’ refers to the same underlying object as ‘p1’ but acts like a 3D point (`IMP.core.XYZ` class)

`set_coordinates()` is a *method* of the XYZ class and `IMP.algebra.Vector3D` represents a 3D vector or coordinate.

A single particle can be decorated multiple times (e.g. can be a 3D point and also have mass, be part of a bond, and have a parent, such as a residue).

In [ ]:
# "Decorate" the Particles with x,y,z attributes
# (point-like particles)
d1 = IMP.core.XYZ.setup_particle(m, p1)
d2 = IMP.core.XYZ.setup_particle(m, p2)

# Use some XYZ-specific functionality (set coordinates)
d1.set_coordinates(IMP.algebra.Vector3D(10.0, 10.0, 10.0))
d2.set_coordinates(IMP.algebra.Vector3D(-10.0, -10.0, -10.0))
print(d1, d2)

## Single-particle restraints

A `Restraint` is a term in our scoring function, just a function of one or more particles.

`IMP.core.SingletonRestraint` applies a `SingletonScore` to a single particle (p1 in this case).

In turn, `DistanceToSingletonScore` calculates the Cartesian distance between a fixed point and p1, then uses a `UnaryFunction` to weight that distance. `Harmonic` is a unary function that applies a simple harmonic spring.

In this way, we can very flexibly build our
scoring function from basic building blocks.

In [ ]:
# Harmonically restrain p1 to be zero distance
# from the origin
f = IMP.core.Harmonic(0.0, 1.0)
s = IMP.core.DistanceToSingletonScore(f,
                      IMP.algebra.Vector3D(0., 0., 0.))
r1 = IMP.core.SingletonRestraint(m, s, p1)

## Two-particle restraints

Similarly, we make another `Restraint` called ‘r2’ that restrains the distance between two particles.

Usually distances are considered to be angstroms but this isn’t required or enforced.

Note that the `IMP.core` module provides simple ‘building block’ restraints.

More complex restraints to handle specific types of input data are found in other modules (e.g. the `IMP.em` and `IMP.saxs` modules provide restraints to handle EM and SAXS data respectively).

In [ ]:
# Harmonically restrain p1 and p2 to be distance
# 5.0 apart
f = IMP.core.Harmonic(5.0, 1.0)
s = IMP.core.DistancePairScore(f)
r2 = IMP.core.PairRestraint(m, s, (p1, p2))

## Sampling

Finally, we make a simple scoring function ‘sf’ that’s just the sum of the two harmonic restraints.

We find the minimum of the function using up to 50 steps of conjugate gradients.

At each step the algorithm will try to reduce the value of the scoring function by changing the coordinates of d1 and/or d2.

In [ ]:
# Optimize the x,y,z coordinates of both particles
# with conjugate gradients
sf = IMP.core.RestraintsScoringFunction([r1, r2],
                                    "scoring function")
d1.set_coordinates_are_optimized(True)
d2.set_coordinates_are_optimized(True)
o = IMP.core.ConjugateGradients(m)
o.set_scoring_function(sf)
o.optimize(50)
print(d1, d2)

As expected, the first particle is pulled to the origin and the second to be distance 5.0 away.

# Domain-specific tools

The C++/Python library is very flexible and has been used for specialist modeling, e.g. https://salilab.org/spb/

In practice though, scripts for most modeling problems would be too long and unwieldy to write this way.

Instead, IMP provides a number of domain-specific tools for performing specialized tasks.

One such tool is FoXS which, given an experimental SAXS profile and a 3D model,
- Calculates the theoretical profile of the model
- Fits the two profiles together and reports a fit value, χ

Here we’ll use FoXS to improve the structure of the C terminal domain of Nup133, one of the subunits of the Nup84 subcomplex of the NPC. First, let's get the inputs for this example, which are included in the IMP distribution, and copy them to our local directory:

In [ ]:
import IMP.foxs
foxs_path = IMP.foxs.get_example_path('nup133')
!cp {foxs_path}/* .

Inputs are 3KFO.pdb, the X-ray crystal structure of the C terminal domain of Nup133, in PDB format; and 23922_merge.dat, the experimental SAXS profile of the same structure (simple table of intensity vs. angle, plotted here for clarity). We'll look at the other PDB file later.

In [ ]:
!ls

Running FoXS is simple; we just give it the PDB file and the profile:

In [ ]:
!foxs 3KFO.pdb 23922_merge.dat

i.e. quality of fit (χ<sup>2</sup>) is 8.76 (smaller is better, so this is not great)

## Improving the fit

The PDB is not the whole truth (as can be seen from REMARK records in the PDB file):
- proteolysis removed residues 881-943 (REMARK 999) so these weren’t seen in either experiment
- the X-ray experiment was unable to resolve residues 944, 945, and 1159-1165 (REMARK 465)
- 16 other residues in the X-ray experiment had unresolved side chains (REMARK 470)

We can resolve these issues by filling in the missing residues and side chains, e.g. with MODELLER, to yield 3KFO-fill.B99990005.pdb

FoXS is run in the same way as before, just on the model:

In [ ]:
!foxs 3KFO-fill.B99990005.pdb 23922_merge.dat

i.e. quality of fit (χ<sup>2</sup>) is 1.31, much improved.

# PMI

PMI spans the gap between specialist applications and direct use of the IMP library. It is a “top-down” rather than “bottom-up” IMP module designed for modeling of complete biological systems.

An introduction to integrative modeling with PMI can be found at https://integrativemodeling.org/tutorials/rnapolii_stalk/. If desired, the Python scripts in that tutorial can also be run in this Google Colab environment. The cell below will obtain the tutorial files and run the main modeling script:

In [ ]:
!git clone https://github.com/salilab/imp_tutorial.git
!cd imp_tutorial/rnapolii/modeling/
!python3 modeling.py --test

# Further reading<a id="further"></a>

More tutorials on using IMP are available at [the IMP web site](https://integrativemodeling.org/tutorials/).